<a href="https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MalikZeeshan1122/FlyRank-ML-Internship-Starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The playbook converts the model/baseline ranking into a human-review queue. Pages with stronger measured opportunity signals are reviewed first, but the score is treated as decision-support rather than an automatic instruction to change content.

Reason codes describe the main observed condition behind the recommendation:

* **HIGH_OPPORTUNITY** — high relative action score and suitable for priority review.
* **MEDIUM_OPPORTUNITY** — moderate relative score and suitable for secondary review.
* **LOW_OPPORTUNITY** — lower relative score; normally lower priority.
* **DATA_LIMIT** — insufficient or unreliable information for a confident recommendation.

The recommended action is **review first**, not automatically rewrite or publish changes.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Locate repository
# --------------------------------------------------

repo_candidates = [
    "/content/flyrank-ml-internship-starter",
    "flyrank-ml-internship-starter",
    "."
]

repo_path = None

for path in repo_candidates:
    if os.path.exists(
        os.path.join(path, "data/raw/content_refresh_anonymized.csv")
    ):
        repo_path = path
        break

# Clone only if repository was not found
if repo_path is None:
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
    repo_path = "/content/flyrank-ml-internship-starter"

print("Repository:", repo_path)

# --------------------------------------------------
# 2. Load dataset
# --------------------------------------------------

csv_path = os.path.join(
    repo_path,
    "data/raw/content_refresh_anonymized.csv"
)

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"Dataset not found: {csv_path}"
    )

df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)

# --------------------------------------------------
# 3. Load ML-07 baseline if it exists
# --------------------------------------------------

baseline_path = os.path.join(
    repo_path,
    "work/outputs/baseline_action_score.csv"
)

if os.path.exists(baseline_path):

    print("\nExisting ML-07 baseline found.")
    baseline = pd.read_csv(baseline_path)

else:

    print("\nML-07 baseline not found.")
    print("Recreating the baseline score...")

    # Never use outcome/label fields as features
    excluded_fields = {
        "trend_direction",
        "trend_pct"
    }

    numeric_cols = df.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    score_features = [
        col
        for col in numeric_cols
        if col not in excluded_fields
    ]

    score_parts = []

    for col in score_features:

        series = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        if series.notna().sum() == 0:
            continue

        series = series.fillna(series.median())

        minimum = series.min()
        maximum = series.max()

        if maximum != minimum:

            normalized = (
                series - minimum
            ) / (
                maximum - minimum
            )

            score_parts.append(normalized)

    if not score_parts:
        raise ValueError(
            "No usable numeric features available."
        )

    score = pd.concat(
        score_parts,
        axis=1
    ).mean(axis=1)

    baseline = pd.DataFrame({
        "row_id": range(len(df)),
        "action_score": score
    })

    baseline["rank"] = (
        baseline["action_score"]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )

    q75 = baseline["action_score"].quantile(0.75)
    q25 = baseline["action_score"].quantile(0.25)

    baseline["reason_code"] = np.select(
        [
            baseline["action_score"] >= q75,
            baseline["action_score"] <= q25
        ],
        [
            "HIGH_OPPORTUNITY",
            "LOW_OPPORTUNITY"
        ],
        default="MEDIUM_OPPORTUNITY"
    )

    os.makedirs(
        os.path.dirname(baseline_path),
        exist_ok=True
    )

    baseline = baseline.sort_values("rank")

    baseline.to_csv(
        baseline_path,
        index=False
    )

    print(
        "Baseline recreated and saved to:",
        baseline_path
    )

print("\nBaseline rows:", len(baseline))
print("\nTop 5 baseline records:")

display(
    baseline.head(5)
)

Repository: /content/flyrank-ml-internship-starter
Dataset shape: (30000, 44)

ML-07 baseline not found.
Recreating the baseline score...
Baseline recreated and saved to: /content/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv

Baseline rows: 30000

Top 5 baseline records:


,row_id,action_score,rank,reason_code
29400,29400,0.423575,1,HIGH_OPPORTUNITY
10741,10741,0.414457,2,HIGH_OPPORTUNITY
21565,21565,0.365559,3,HIGH_OPPORTUNITY
13537,13537,0.338706,4,HIGH_OPPORTUNITY
16811,16811,0.335106,5,HIGH_OPPORTUNITY


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

The playbook is intended for SEO/content teams to prioritize which anonymized pages should receive human review first. It helps organize a large content set using measured signals from the available search dataset.

The output is **decision-support**, not an automatic content-generation or publishing system.

The recommendations are limited by the available data, the defined proxy target, the validation design, and possible changes in search behavior over time. A high score does not mean that refreshing a page will definitely increase traffic, rankings, or conversions.

The system should not be interpreted as predicting or explaining Google's ranking algorithm.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended use: prioritize pages for human content review.")
print("Decision type: review prioritization.")
print("Automation level: decision-support only.")

print("\nKnown limits:")
limits = [
    "A high score does not establish causality.",
    "The model does not predict Google's ranking algorithm.",
    "Historical search patterns may become stale.",
    "The dataset is anonymized and may omit useful business context.",
    "Human review is required before taking content actions."
]

for item in limits:
    print("-", item)


Intended use: prioritize pages for human content review.
Decision type: review prioritization.
Automation level: decision-support only.

Known limits:
- A high score does not establish causality.
- The model does not predict Google's ranking algorithm.
- Historical search patterns may become stale.
- The dataset is anonymized and may omit useful business context.
- Human review is required before taking content actions.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review + the no-go list

Before acting on a recommendation, a human reviewer should check the page's search intent, current content quality, factual accuracy, topical relevance, recent changes, and whether the observed performance pattern is meaningful enough to justify an intervention.

The model should never independently decide to publish, delete, rewrite, redirect, or make factual claims about content.

The following actions remain human-controlled:

* publishing or deleting content;
* changing important factual or legal claims;
* changing URLs, redirects, or canonicalization;
* making large-scale site-wide changes;
* deciding that a page should be abandoned;
* interpreting business or brand implications.

The model provides prioritization evidence; the final decision remains with a qualified reviewer.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Human review checklist

human_review_checks = [
    "Check search intent and topical relevance.",
    "Check whether the content is factually accurate.",
    "Check whether the page has recently changed.",
    "Check whether the observed traffic pattern is meaningful.",
    "Check whether the recommendation fits the business context.",
    "Check whether the proposed change has unintended SEO risks."
]

no_go_actions = [
    "Automatic publishing",
    "Automatic deletion",
    "Automatic URL/redirect changes",
    "Automatic factual claims",
    "Automatic site-wide content changes",
    "Automatic abandonment of pages"
]

print("HUMAN REVIEW CHECKS")
for item in human_review_checks:
    print("-", item)

print("\nNO-GO AUTOMATIONS")
for item in no_go_actions:
    print("-", item)


HUMAN REVIEW CHECKS
- Check search intent and topical relevance.
- Check whether the content is factually accurate.
- Check whether the page has recently changed.
- Check whether the observed traffic pattern is meaningful.
- Check whether the recommendation fits the business context.
- Check whether the proposed change has unintended SEO risks.

NO-GO AUTOMATIONS
- Automatic publishing
- Automatic deletion
- Automatic URL/redirect changes
- Automatic factual claims
- Automatic site-wide content changes
- Automatic abandonment of pages


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring / retrain triggers

The recommendations should be monitored because the relationship between page characteristics and search performance can change.

A review or retraining cycle should be considered when:

1. model performance falls materially compared with the validated baseline;
2. Precision@K or the chosen ranking metric deteriorates;
3. the distribution of important input features changes substantially;
4. the proportion of missing values increases;
5. the distribution of trend outcomes changes;
6. search behavior or the underlying data collection process changes;
7. repeated human review shows that high-ranked recommendations are frequently unsuitable.

These are monitoring signals rather than automatic proof that the model has failed.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Basic monitoring checks

print("MONITORING SNAPSHOT")
print("- Dataset rows:", len(df))

missing_rate = (
    df.isna().mean()
    .sort_values(ascending=False)
    .head(10)
)

print("\nTop missing-value rates:")
display(missing_rate.to_frame("missing_rate"))

if "trend_direction" in df.columns:
    print("\nTrend distribution:")
    display(
        df["trend_direction"]
        .value_counts(dropna=False)
        .to_frame("count")
    )

print("\nSuggested retrain/review triggers:")
triggers = [
    "Material decline in validation ranking performance",
    "Feature distribution drift",
    "Increase in missing-value rates",
    "Change in target/trend distribution",
    "Change in data collection or measurement process",
    "Repeated human-review disagreement with high-ranked recommendations"
]

for trigger in triggers:
    print("-", trigger)

MONITORING SNAPSHOT
- Dataset rows: 30000

Top missing-value rates:


,missing_rate
provider_used,0.714600
word_count,0.256633
char_count,0.256633
word_count_tier,0.256633
char_count_tier,0.256633
model_used,0.191100
trend_pct,0.112933
competition_level,0.087000
search_volume,0.082267
cpc,0.082267



Trend distribution:


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



Suggested retrain/review triggers:
- Material decline in validation ranking performance
- Feature distribution drift
- Increase in missing-value rates
- Change in target/trend distribution
- Change in data collection or measurement process
- Repeated human-review disagreement with high-ranked recommendations


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The final ranked queue is exported to `work/outputs/` so that the same decision-support artifact can be reused consistently in the capstone report.

The exported file contains the ranking, score, reason code, recommended review action, and confidence/limitation notes. It does not contain client names, URLs, private queries, or other identifying information.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# 5. EXPORTS FOR THE PAPER — ROBUST VERSION
# ============================================================

import os
import pandas as pd
import numpy as np

# Make sure baseline exists
if "baseline" not in globals():
    raise RuntimeError(
        "The baseline dataframe is not available. "
        "Run the ML-07 baseline/loading cell first."
    )

# Make a fresh copy
playbook = baseline.copy()

# ------------------------------------------------------------
# Create reason codes if they are missing
# ------------------------------------------------------------

if "reason_code" not in playbook.columns:

    q75 = playbook["action_score"].quantile(0.75)
    q25 = playbook["action_score"].quantile(0.25)

    playbook["reason_code"] = np.select(
        [
            playbook["action_score"] >= q75,
            playbook["action_score"] <= q25
        ],
        [
            "HIGH_OPPORTUNITY",
            "LOW_OPPORTUNITY"
        ],
        default="MEDIUM_OPPORTUNITY"
    )

# ------------------------------------------------------------
# Create actions
# ------------------------------------------------------------

action_map = {
    "HIGH_OPPORTUNITY": "Priority human review",
    "MEDIUM_OPPORTUNITY": "Secondary human review",
    "LOW_OPPORTUNITY": "Monitor / lower priority",
    "DATA_LIMIT": "Manual review"
}

playbook["action"] = (
    playbook["reason_code"]
    .map(action_map)
    .fillna("Manual review")
)

# ------------------------------------------------------------
# Create confidence notes
# ------------------------------------------------------------

confidence_map = {
    "HIGH_OPPORTUNITY":
        "Higher relative score; prioritize human review.",

    "MEDIUM_OPPORTUNITY":
        "Moderate relative score; review after higher-priority items.",

    "LOW_OPPORTUNITY":
        "Lower relative score; generally monitor.",

    "DATA_LIMIT":
        "Insufficient information for a confident recommendation."
}

playbook["confidence_note"] = (
    playbook["reason_code"]
    .map(confidence_map)
    .fillna("Limited signal.")
)

# ------------------------------------------------------------
# Create limitation note
# ------------------------------------------------------------

playbook["what_would_make_it_wrong"] = (
    "Noisy or incomplete measurements, changed search behavior, "
    "recent page changes, or missing business context."
)

# ------------------------------------------------------------
# Make sure rank exists
# ------------------------------------------------------------

if "rank" not in playbook.columns:

    playbook["rank"] = (
        playbook["action_score"]
        .rank(
            method="first",
            ascending=False
        )
        .astype(int)
    )

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

output_dir = os.path.join(
    repo_path,
    "work",
    "outputs"
)

os.makedirs(
    output_dir,
    exist_ok=True
)

output_path = os.path.join(
    output_dir,
    "content_action_playbook.csv"
)

export_columns = [
    "rank",
    "row_id",
    "action_score",
    "reason_code",
    "action",
    "confidence_note",
    "what_would_make_it_wrong"
]

# Verify columns before exporting
missing_columns = [
    col for col in export_columns
    if col not in playbook.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

final_queue = (
    playbook[export_columns]
    .sort_values("rank")
    .reset_index(drop=True)
)

final_queue.to_csv(
    output_path,
    index=False
)

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("✅ ML-10 EXPORT SUCCESSFUL")
print()
print("Rows exported:", len(final_queue))
print("Columns:", final_queue.columns.tolist())
print("Output:", output_path)

print("\nTop 20 action queue:")
display(final_queue.head(20))

✅ ML-10 EXPORT SUCCESSFUL

Rows exported: 30000
Columns: ['rank', 'row_id', 'action_score', 'reason_code', 'action', 'confidence_note', 'what_would_make_it_wrong']
Output: /content/flyrank-ml-internship-starter/work/outputs/content_action_playbook.csv

Top 20 action queue:


,rank,row_id,action_score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,29400,0.423575,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."
1,2,10741,0.414457,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."
2,3,21565,0.365559,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."
3,4,13537,0.338706,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."
4,5,16811,0.335106,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."
5,6,17812,0.332228,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."
6,7,21819,0.318423,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."
7,8,14178,0.318262,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."
8,9,14234,0.298699,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."
9,10,2797,0.297626,HIGH_OPPORTUNITY,Priority human review,Higher relative score; prioritize human review.,"Noisy or incomplete measurements, changed sear..."


In [9]:
# ============================================================
# FINAL SELF-CHECK
# ============================================================

required_columns = [
    "rank",
    "row_id",
    "action_score",
    "reason_code",
    "action",
    "confidence_note",
    "what_would_make_it_wrong"
]

assert len(final_queue) == len(df)

assert all(
    column in final_queue.columns
    for column in required_columns
)

assert final_queue["rank"].is_unique

assert os.path.exists(output_path)

print("===================================")
print("ML-10 SELF-CHECK: PASSED ✅")
print("===================================")
print("Dataset rows:", len(df))
print("Queue rows:", len(final_queue))
print("Required columns: present")
print("Rank uniqueness: passed")
print("Output file: exists")
print()
print(output_path)

ML-10 SELF-CHECK: PASSED ✅
Dataset rows: 30000
Queue rows: 30000
Required columns: present
Rank uniqueness: passed
Output file: exists

/content/flyrank-ml-internship-starter/work/outputs/content_action_playbook.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.